# Olist E-commerce: Sales, Customer & Delivery Analytics
**Author:** Arnav Shirke

**Business questions**
1. How is revenue trending, and which categories and states drive it?
2. Who are our best customers, and how many are repeat buyers? (RFM segmentation)
3. Does late delivery hurt customer satisfaction, and where is delivery weakest?

**What this script does end to end:** loads the 9 Olist tables, cleans them, builds analysis-ready
tables for Power BI, runs the analysis, saves charts, writes an interactive HTML dashboard,
and auto-generates insights, a README and resume bullets from the *actual* numbers.

Run:  `python olist_pipeline.py`   (downloads the data with kagglehub on first run)

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

if "ipykernel" not in sys.modules:      # script mode: no popup windows
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

OUT = Path(os.environ.get("OUT_DIR", "output"))
IMG, PBI = OUT / "images", OUT / "powerbi_data"
for p in (IMG, PBI):
    p.mkdir(parents=True, exist_ok=True)

# The Olist data is in Brazilian Real (BRL). All money values are converted to INR once, right after loading.
# Change the rate here (or run:  set BRL_TO_INR=18.9) to refresh it. Market rate in Sept 2026 was about 18.6.
BRL_TO_INR = float(os.environ.get("BRL_TO_INR", 18.6))

ACCENT, GOOD, BAD, GREY = "#2563EB", "#16A34A", "#DC2626", "#94A3B8"
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})


def get_data_dir() -> Path:
    d = os.environ.get("OLIST_DATA_DIR")           # optional: point to a local folder
    if d and Path(d).is_dir():
        return Path(d)
    import kagglehub
    return Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))


DATA_DIR = get_data_dir()
print("Data folder:", DATA_DIR)

## 1. Load data

In [ ]:
def rd(name, **kw):
    return pd.read_csv(DATA_DIR / name, **kw)


customers = rd("olist_customers_dataset.csv")
orders = rd("olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date"])
items = rd("olist_order_items_dataset.csv")
payments = rd("olist_order_payments_dataset.csv")
reviews = rd("olist_order_reviews_dataset.csv")
products = rd("olist_products_dataset.csv")
sellers = rd("olist_sellers_dataset.csv")
trans = rd("product_category_name_translation.csv")

items["price"] *= BRL_TO_INR
items["freight_value"] *= BRL_TO_INR
payments["payment_value"] *= BRL_TO_INR
print(f"Money converted from BRL to INR at 1 BRL = Rs {BRL_TO_INR}")

for n, d in [("customers", customers), ("orders", orders), ("items", items),
             ("payments", payments), ("reviews", reviews), ("products", products),
             ("sellers", sellers)]:
    print(f"{n:10s} {d.shape}")

## 2. Clean data (and keep a record of what was fixed)

In [ ]:
dq = {}
dq["Total orders in raw data"] = len(orders)
dq["Orders with status 'delivered'"] = int((orders.order_status == "delivered").sum())
dq["Orders with duplicate reviews (kept the average score)"] = int(reviews.order_id.duplicated().sum())
dq["Products with missing category (labelled 'Unknown')"] = int(products.product_category_name.isna().sum())

# English category names, tidy labels
products = products.merge(trans, on="product_category_name", how="left")
products["category"] = (products.product_category_name_english
                        .fillna(products.product_category_name)
                        .fillna("unknown").str.replace("_", " ").str.title())

items = (items.merge(products[["product_id", "category"]], on="product_id", how="left")
              .merge(sellers[["seller_id", "seller_city", "seller_state"]], on="seller_id", how="left"))
items["category"] = items.category.fillna("Unknown")

# Delivered orders only; drop the few delivered orders that have no delivery timestamp
delivered = orders[orders.order_status == "delivered"].copy()
missing_dt = delivered.order_delivered_customer_date.isna().sum()
dq["Delivered orders dropped (no delivery date)"] = int(missing_dt)
delivered = delivered.dropna(subset=["order_delivered_customer_date"])
delivered = delivered.merge(customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
                            on="customer_id", how="left")

# Order-level aggregates
item_agg = items.groupby("order_id").agg(
    revenue=("price", "sum"), freight=("freight_value", "sum"),
    items_count=("order_item_id", "count"), sellers_count=("seller_id", "nunique")).reset_index()
pay_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"), max_installments=("payment_installments", "max")).reset_index()
main_pay = (payments.sort_values("payment_value", ascending=False)
            .drop_duplicates("order_id")[["order_id", "payment_type"]])
rev_agg = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()
main_cat = (items.sort_values("price", ascending=False)
            .drop_duplicates("order_id")[["order_id", "category"]])

fact = (delivered.merge(item_agg, on="order_id", how="left")
        .merge(pay_agg, on="order_id", how="left")
        .merge(main_pay, on="order_id", how="left")
        .merge(rev_agg, on="order_id", how="left")
        .merge(main_cat, on="order_id", how="left"))
fact[["revenue", "freight"]] = fact[["revenue", "freight"]].fillna(0)
fact["payment_value"] = fact.payment_value.fillna(fact.revenue + fact.freight)
fact["category"] = fact.category.fillna("Unknown")
fact["payment_type"] = fact.payment_type.fillna("unknown")

# Derived columns
fact["purchase_date"] = fact.order_purchase_timestamp.dt.normalize()
fact["purchase_month"] = fact.order_purchase_timestamp.dt.to_period("M").dt.to_timestamp()
fact["delivery_days"] = (fact.order_delivered_customer_date - fact.order_purchase_timestamp).dt.total_seconds() / 86400
fact["delay_days"] = (fact.order_delivered_customer_date - fact.order_estimated_delivery_date).dt.total_seconds() / 86400
fact["is_late"] = (fact.delay_days > 0).astype(int)
fact["delay_bucket"] = pd.cut(fact.delay_days, [-np.inf, 0, 3, 7, np.inf],
                              labels=["On time / early", "1-3 days late", "4-7 days late", "8+ days late"]).astype(str)
dq["Final orders used in analysis"] = len(fact)
print(pd.Series(dq))

## 3. RFM customer segmentation
Olist has very few repeat buyers, so a pure quintile-based Frequency score is meaningless
(95%+ of customers have 1 order). Frequency is therefore rule-based, Recency and Monetary use quintiles.

In [ ]:
snapshot = fact.order_purchase_timestamp.max().normalize() + pd.Timedelta(days=1)
rfm = fact.groupby("customer_unique_id").agg(
    last_purchase=("order_purchase_timestamp", "max"), first_purchase=("order_purchase_timestamp", "min"),
    frequency=("order_id", "nunique"), monetary=("payment_value", "sum"),
    customer_state=("customer_state", "last")).reset_index()
rfm["recency_days"] = (snapshot - rfm.last_purchase.dt.normalize()).dt.days
rfm["r_score"] = pd.qcut(rfm.recency_days.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["m_score"] = pd.qcut(rfm.monetary.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["f_score"] = np.where(rfm.frequency >= 3, 5, np.where(rfm.frequency == 2, 3, 1))

f, r, m = rfm.frequency, rfm.r_score, rfm.m_score
SEG_ORDER = ["Champions", "Loyal Repeat", "New High-Value", "New Customers",
             "Need Attention", "At-Risk High-Value", "Lost / Hibernating"]
rfm["segment"] = np.select(
    [(f >= 2) & (r >= 4), (f >= 2),
     (f == 1) & (r >= 4) & (m >= 4), (f == 1) & (r >= 4),
     (f == 1) & (r == 3), (f == 1) & (r <= 2) & (m >= 4)],
    SEG_ORDER[:6], default=SEG_ORDER[6])

seg = (rfm.groupby("segment").agg(customers=("customer_unique_id", "count"), revenue=("monetary", "sum"),
                                  avg_spend=("monetary", "mean")).reindex(SEG_ORDER).fillna(0))
seg["pct_customers"] = 100 * seg.customers / seg.customers.sum()
seg["pct_revenue"] = 100 * seg.revenue / seg.revenue.sum()
print(seg.round(1))

## 4. Export analysis-ready tables for Power BI

In [ ]:
dim_date = pd.DataFrame({"date": pd.date_range(fact.purchase_date.min(), fact.purchase_date.max(), freq="D")})
dim_date["year"] = dim_date.date.dt.year
dim_date["quarter"] = "Q" + dim_date.date.dt.quarter.astype(str)
dim_date["month_num"] = dim_date.date.dt.month
dim_date["month_name"] = dim_date.date.dt.strftime("%b")
dim_date["year_month"] = dim_date.date.dt.strftime("%Y-%m")
dim_date["weekday_num"] = dim_date.date.dt.dayofweek + 1
dim_date["weekday_name"] = dim_date.date.dt.strftime("%a")
dim_date["is_weekend"] = (dim_date.weekday_num >= 6).astype(int)

fact_orders = fact[["order_id", "customer_unique_id", "customer_city", "customer_state", "purchase_date",
                    "purchase_month", "order_status", "category", "payment_type", "max_installments",
                    "items_count", "sellers_count", "revenue", "freight", "payment_value", "review_score",
                    "delivery_days", "delay_days", "is_late", "delay_bucket"]].copy()
fact_order_items = (items.merge(fact[["order_id", "purchase_date"]], on="order_id", how="inner")
                    [["order_id", "order_item_id", "product_id", "seller_id", "seller_city", "seller_state",
                      "category", "price", "freight_value", "purchase_date"]])
dim_customer_rfm = rfm[["customer_unique_id", "customer_state", "recency_days", "frequency", "monetary",
                        "r_score", "f_score", "m_score", "segment"]]

for name, df in [("dim_date", dim_date), ("fact_orders", fact_orders),
                 ("fact_order_items", fact_order_items), ("dim_customer_rfm", dim_customer_rfm)]:
    df.round(2).to_csv(PBI / f"{name}.csv", index=False)
    print(f"saved {name}.csv  rows={len(df):,}")

## 5. Analysis

In [ ]:
def money(x):
    a = abs(x)
    if a >= 1e7:
        return f"₹{x/1e7:.2f} Cr"
    if a >= 1e5:
        return f"₹{x/1e5:.2f} L"
    return f"₹{x:,.0f}"


n_orders = fact.order_id.nunique()
kpi = dict(orders=n_orders, customers=fact.customer_unique_id.nunique(), revenue=fact.revenue.sum(),
           aov=fact.revenue.sum() / n_orders, review=fact.review_score.mean(),
           delivery=fact.delivery_days.mean(), late=fact.is_late.mean() * 100)

monthly = fact.groupby("purchase_month").agg(orders=("order_id", "nunique"), revenue=("revenue", "sum")).reset_index()
trend = monthly[(monthly.purchase_month >= "2017-01-01") & (monthly.purchase_month <= "2018-08-01")]
if len(trend) < 6:
    trend = monthly
peak = trend.loc[trend.revenue.idxmax()]


def ytd(year):
    d = monthly[(monthly.purchase_month.dt.year == year) & (monthly.purchase_month.dt.month <= 8)]
    return d.revenue.sum(), len(d)


(r17, n17), (r18, n18) = ytd(2017), ytd(2018)
yoy = (r18 / r17 - 1) * 100 if (n17 == 8 and n18 == 8 and r17 > 0) else None

cat_rev = items.merge(fact[["order_id"]], on="order_id").groupby("category").price.sum().sort_values(ascending=False)
top5_share = 100 * cat_rev.head(5).sum() / cat_rev.sum()
state_rev = fact.groupby("customer_state").revenue.sum().sort_values(ascending=False)
top_state, top_state_share = state_rev.index[0], 100 * state_rev.iloc[0] / state_rev.sum()

min_orders = max(20, int(len(fact) * 0.003))
state_ops = (fact.groupby("customer_state").agg(orders=("order_id", "count"), avg_days=("delivery_days", "mean"),
                                                late_pct=("is_late", "mean"), review=("review_score", "mean"))
             .query("orders >= @min_orders"))
state_ops["late_pct"] *= 100
worst_states = state_ops.sort_values("avg_days", ascending=False).head(10)

rev_late = fact.groupby("is_late").review_score.mean()
bucket_order = ["On time / early", "1-3 days late", "4-7 days late", "8+ days late"]
bucket_rev = fact.groupby("delay_bucket").review_score.mean().reindex(bucket_order)
bucket_cnt = fact.groupby("delay_bucket").size().reindex(bucket_order)

repeat_pct = 100 * (rfm.frequency > 1).mean()
top20 = rfm.sort_values("monetary", ascending=False).head(int(len(rfm) * 0.2))
top20_share = 100 * top20.monetary.sum() / rfm.monetary.sum()
card_pct = 100 * (fact.payment_type == "credit_card").mean()
print({k: round(v, 2) for k, v in kpi.items()})

## 6. Charts (PNG for GitHub + Plotly for the interactive dashboard)

In [ ]:
def style(fig, title, h=380):
    fig.update_layout(template="plotly_white", height=h, margin=dict(l=10, r=20, t=55, b=30),
                      title=dict(text=title, x=0.01, font=dict(size=15)),
                      font=dict(family="Segoe UI, Arial, sans-serif", size=12),
                      legend=dict(orientation="h", y=-0.15))
    return fig


def barh_png(s, title, fname, fmt="{:,.0f}", color=ACCENT):
    s = s.sort_values()
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.barh(s.index.astype(str), s.values, color=color)
    for i, v in enumerate(s.values):
        ax.text(v, i, " " + fmt.format(v), va="center", fontsize=8)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.margins(x=0.18)
    fig.tight_layout()
    fig.savefig(IMG / fname)
    plt.close(fig)


def barh_fig(s, title, fmt=",.0f", color=ACCENT):
    s = s.sort_values()
    fig = go.Figure(go.Bar(x=s.values, y=s.index.astype(str), orientation="h", marker_color=color,
                           text=[format(v, fmt) for v in s.values], textposition="outside", cliponaxis=False))
    return style(fig, title)


figs = {}

# 1 Monthly trend
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(trend.purchase_month, trend.orders, width=20, color=GREY, alpha=0.35, label="Orders")
ax.set_ylabel("Orders")
ax2 = ax.twinx()
ax2.plot(trend.purchase_month, trend.revenue / 1e5, color=ACCENT, marker="o", label="Revenue")
ax2.set_ylabel("Revenue (₹ lakh)")
ax.set_title("Monthly revenue and orders", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(IMG / "01_monthly_revenue.png"); plt.close(fig)
f1 = make_subplots(specs=[[{"secondary_y": True}]])
f1.add_trace(go.Bar(x=trend.purchase_month, y=trend.orders, name="Orders", marker_color=GREY, opacity=0.4), secondary_y=False)
f1.add_trace(go.Scatter(x=trend.purchase_month, y=trend.revenue, name="Revenue (₹)", mode="lines+markers",
                        line=dict(color=ACCENT, width=3)), secondary_y=True)
figs["trend"] = style(f1, "Monthly revenue and orders")

# 2 Categories
top_cat = cat_rev.head(10)
barh_png(top_cat, "Top 10 categories by revenue (₹)", "02_top_categories.png")
figs["cat"] = barh_fig(top_cat, "Top 10 categories by revenue (₹)")

# 3 Delay vs review
cols = [GOOD, "#F59E0B", "#F97316", BAD]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(bucket_order, bucket_rev.values, color=cols)
for i, v in enumerate(bucket_rev.values):
    ax.text(i, v + 0.03, f"{v:.2f}", ha="center", fontweight="bold")
ax.set_ylim(0, 5); ax.set_ylabel("Average review score (1-5)")
ax.set_title("Late delivery vs customer review score", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(IMG / "03_delay_vs_review.png"); plt.close(fig)
figs["delay"] = style(go.Figure(go.Bar(x=bucket_order, y=bucket_rev.values, marker_color=cols,
                                       text=[f"{v:.2f}" for v in bucket_rev.values], textposition="outside")),
                      "Late delivery vs customer review score")
figs["delay"].update_yaxes(range=[0, 5], title="Avg review score")

# 4 States
top_states = state_rev.head(10)
barh_png(top_states, "Top 10 customer states by revenue (₹)", "04_top_states.png")
figs["state"] = barh_fig(top_states, "Top 10 customer states by revenue (₹)")

# 5 RFM
s_ = seg.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(s_))
ax.barh(y + 0.2, s_.pct_customers, height=0.38, color=GREY, label="% of customers")
ax.barh(y - 0.2, s_.pct_revenue, height=0.38, color=ACCENT, label="% of revenue")
ax.set_yticks(y); ax.set_yticklabels(s_.index); ax.legend(frameon=False)
ax.set_title("RFM segments: share of customers vs revenue", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(IMG / "05_rfm_segments.png"); plt.close(fig)
f5 = go.Figure()
f5.add_bar(x=seg.index, y=seg.pct_customers, name="% of customers", marker_color=GREY)
f5.add_bar(x=seg.index, y=seg.pct_revenue, name="% of revenue", marker_color=ACCENT)
f5.update_layout(barmode="group")
figs["rfm"] = style(f5, "RFM segments: share of customers vs revenue")

# 6 Slowest states
ws = worst_states.avg_days
barh_png(ws, "Slowest states: average delivery days", "06_slowest_states.png", fmt="{:.1f}", color=BAD)
figs["slow"] = barh_fig(ws, "Slowest states: average delivery days", fmt=".1f", color=BAD)
print("charts saved")

## 7. Insights, README, resume bullets and interactive dashboard (auto-generated from the numbers)

In [ ]:
worst3 = ", ".join(worst_states.index[:3])
gap = rev_late.get(0, np.nan) - rev_late.get(1, np.nan)
seg_top = seg.loc[["New High-Value", "At-Risk High-Value"], "customers"].sum()
only = "only " if repeat_pct < 20 else ""
one_time_txt = ", so almost all revenue comes from one-time buyers" if repeat_pct < 10 else ""
geo_txt = ", which shows heavy geographic concentration" if top_state_share > 30 else ""
yoy_txt = f"Revenue for Jan-Aug 2018 was {yoy:+.0f}% versus the same months of 2017." if yoy is not None else ""

insights = [
    f"The business generated **{money(kpi['revenue'])}** from **{kpi['orders']:,}** delivered orders and "
    f"**{kpi['customers']:,}** unique customers (average order value **{money(kpi['aov'])}**). {yoy_txt}",
    f"Revenue peaked in **{peak.purchase_month:%b %Y}** at **{money(peak.revenue)}**; stock and logistics "
    f"capacity should be planned ahead of peak months.",
    f"**Late delivery hurts satisfaction:** on-time orders average **{rev_late.get(0, np.nan):.2f}** stars versus "
    f"**{rev_late.get(1, np.nan):.2f}** for late orders (a gap of {gap:.2f}). Orders 8+ days late average "
    f"**{bucket_rev.iloc[3]:.2f}**. Overall, **{kpi['late']:.1f}%** of orders arrive after the promised date.",
    f"{only.capitalize()}**{repeat_pct:.1f}%** of customers ever bought a second time{one_time_txt}. "
    f"The top 20% of customers contribute **{top20_share:.0f}%** of revenue.",
    f"The top 5 categories account for **{top5_share:.0f}%** of revenue (led by **{cat_rev.index[0]}**). "
    f"**{top_state}** alone contributes **{top_state_share:.0f}%**{geo_txt}.",
    f"Delivery is slowest in **{worst3}** (up to **{worst_states.avg_days.iloc[0]:.0f} days** on average vs "
    f"**{kpi['delivery']:.1f}** overall).",
    f"**{seg_top:,}** one-time customers sit in the 'New High-Value' or 'At-Risk High-Value' RFM segments. "
    f"These are the best targets for a win-back or second-purchase campaign. Credit card is used on "
    f"**{card_pct:.0f}%** of orders.",
]
recs = [
    f"**Fix delivery before marketing:** audit carriers and promised-date logic in {worst3}; set realistic "
    f"delivery estimates so fewer orders are late. Track late-delivery % as a weekly KPI.",
    f"**Build a repeat-purchase programme:** with a {repeat_pct:.1f}% repeat rate, a post-delivery email/voucher "
    f"flow for 'New High-Value' and 'New Customers' (30-60 days after purchase) is a low-cost way to grow revenue.",
    f"**Win back high-value lapsed customers:** target the 'At-Risk High-Value' segment with a personalised offer in "
    f"its top categories.",
    f"**Plan for peaks:** build stock and carrier capacity before {peak.purchase_month:%B}, the strongest month.",
    f"**Reduce geographic dependence:** recruit sellers and improve fulfilment for states outside {top_state} so "
    f"delivery times there fall.",
]
limits = ("Findings are associations from observational data (for example, late delivery is linked with lower "
          "ratings but other factors may also play a part). The source data is in Brazilian Real; all values are "
          f"converted to INR at a fixed 1 BRL = ₹{BRL_TO_INR} for illustration (SQL outputs on the raw data are in BRL). "
          "Revenue uses item price, excluding freight. Early 2017 and late 2018 months are partial and excluded from trend charts.")

(OUT / "insights.md").write_text(
    "# Key insights\n\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(insights)) +
    "\n\n# Recommendations\n\n" + "\n".join(f"{i+1}. {t}" for i, t in enumerate(recs)) +
    f"\n\n_Note: {limits}_\n", encoding="utf-8")

(OUT / "resume_bullets.md").write_text(f"""# Resume bullets (numbers come from your analysis)

**E-commerce Sales & Customer Segmentation Analysis** | SQL, Python (pandas), Power BI, DAX | GitHub: [link]
- Analysed **{kpi['orders']:,} delivered orders** across **8 relational tables** using SQL (joins, CTEs, window functions) and Python to uncover revenue trends, category and regional performance (≈{money(kpi['revenue'])} total revenue, converted from BRL).
- Segmented **{kpi['customers']:,} customers** with RFM analysis; found {only}**{repeat_pct:.1f}%** repurchase and that the top 20% of customers drive **{top20_share:.0f}%** of revenue, and recommended targeted retention campaigns.
- Quantified the impact of late delivery on satisfaction (**{rev_late.get(0, np.nan):.2f} vs {rev_late.get(1, np.nan):.2f}** average rating for on-time vs late orders) and identified the slowest regions ({worst3}) for logistics improvement.
- Built a 3-page interactive Power BI dashboard (data model, DAX measures: YoY, MoM, running total) for executive, customer and delivery-performance views.

**LinkedIn one-liner:** End-to-end analytics on {kpi['orders']:,} e-commerce orders (SQL, Python, Power BI): late deliveries cut ratings from {rev_late.get(0, np.nan):.2f} to {rev_late.get(1, np.nan):.2f} stars, and only {repeat_pct:.1f}% of customers reorder.
""", encoding="utf-8")

dq_md = "\n".join(f"- {k}: {v:,}" for k, v in dq.items())
(OUT / "README.md").write_text(f"""# Olist E-commerce: Sales, Customer & Delivery Analytics

End-to-end analytics project on the Olist Brazilian e-commerce dataset using **SQL, Python and Power BI**.

## Business problem
The business wants to grow revenue and retention. This project answers: **how is revenue trending and where does it come from, who are the best customers and how many come back, and does late delivery hurt customer satisfaction?**

## Dataset
[Olist Brazilian E-commerce (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce): about 100K orders (2016-2018) across 9 related tables (orders, items, payments, reviews, customers, sellers, products, geolocation, category translation). Currency: the source data is in Brazilian Real (BRL); values in this project are converted to INR at a fixed 1 BRL = ₹{BRL_TO_INR}.

## Tools
SQL (PostgreSQL), Python (pandas, matplotlib, plotly), Power BI (data modelling, DAX)

## Approach
1. **Data cleaning (Python):** joined 8 tables, standardised categories, handled missing delivery dates, duplicate reviews and missing categories.
{dq_md}
2. **SQL analysis:** revenue trend, MoM growth (window functions), category and seller ranking, delivery performance, repeat vs one-time customers, revenue concentration (see `sql/queries.sql`).
3. **RFM segmentation (Python):** customers scored on Recency, Frequency and Monetary value and grouped into 7 segments.
4. **Dashboard (Power BI):** 3 pages (Executive Summary, Customers, Delivery & Operations) on a star-style data model.

## KPIs
| Revenue | Orders | Customers | Avg order value | Avg review | Avg delivery | Late orders |
|---|---|---|---|---|---|---|
| {money(kpi['revenue'])} | {kpi['orders']:,} | {kpi['customers']:,} | {money(kpi['aov'])} | {kpi['review']:.2f} / 5 | {kpi['delivery']:.1f} days | {kpi['late']:.1f}% |

## Key insights
{chr(10).join(f"{i+1}. {t}" for i, t in enumerate(insights))}

## Recommendations
{chr(10).join(f"{i+1}. {t}" for i, t in enumerate(recs))}

## Dashboard preview
![Dashboard](images/dashboard_screenshot.png)

| | |
|---|---|
| ![](images/01_monthly_revenue.png) | ![](images/03_delay_vs_review.png) |
| ![](images/02_top_categories.png) | ![](images/05_rfm_segments.png) |

## Repository structure
```
sql/          SQL queries
notebooks/    Python analysis (olist_analysis.ipynb)
powerbi_data/ cleaned tables loaded into Power BI
dashboard/    Power BI file (.pbix) and interactive HTML preview
images/       charts and dashboard screenshots
```

## How to run
```
pip install -r requirements.txt
python olist_pipeline.py
```

## Limitations
{limits}

## Author
Arnav Shirke | [LinkedIn](https://www.linkedin.com/) | Data / Business Analyst
""", encoding="utf-8")

kpi_cards = "".join(f'<div class="kpi"><div class="v">{v}</div><div class="l">{l}</div></div>' for l, v in [
    ("Revenue", money(kpi["revenue"])), ("Orders", f"{kpi['orders']:,}"), ("Customers", f"{kpi['customers']:,}"),
    ("Avg order value", money(kpi["aov"])), ("Avg review", f"{kpi['review']:.2f} / 5"),
    ("Avg delivery", f"{kpi['delivery']:.1f} days"), ("Late orders", f"{kpi['late']:.1f}%")])
panels = "".join(f'<div class="card">{figs[k].to_html(full_html=False, include_plotlyjs=("cdn" if i == 0 else False))}</div>'
                 for i, k in enumerate(["trend", "delay", "cat", "state", "rfm", "slow"]))
html = """<!doctype html><html><head><meta charset="utf-8"><title>Olist Sales, Customer & Delivery Dashboard</title>
<style>body{margin:0;background:#F1F5F9;font-family:Segoe UI,Arial,sans-serif;color:#0F172A}
header{background:#0F172A;color:#fff;padding:18px 28px}header h1{margin:0;font-size:22px}header p{margin:4px 0 0;color:#94A3B8;font-size:13px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;padding:16px 28px 0}
.kpi{background:#fff;border-radius:10px;padding:14px;border-left:4px solid #2563EB}.kpi .v{font-size:22px;font-weight:700}.kpi .l{font-size:12px;color:#64748B}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(480px,1fr));gap:14px;padding:16px 28px}
.card{background:#fff;border-radius:10px;padding:6px}.ins{padding:0 28px 28px}.ins div{background:#fff;border-radius:10px;padding:14px 20px}
li{margin:6px 0;font-size:14px;line-height:1.45}</style></head><body>
<header><h1>Olist E-commerce: Sales, Customer & Delivery Dashboard</h1><p>Delivered orders, 2016-2018 | Currency: INR (converted from BRL) | Author: Arnav Shirke</p></header>
<div class="kpis">__KPI__</div><div class="grid">__PANELS__</div>
<div class="ins"><div><h3>Key insights</h3><ol>__INS__</ol></div></div></body></html>"""
ins_html = "".join(f"<li>{t}</li>".replace("**", "") for t in insights)
(OUT / "dashboard.html").write_text(html.replace("__KPI__", kpi_cards).replace("__PANELS__", panels)
                                    .replace("__INS__", ins_html), encoding="utf-8")
print("\nDONE. Open the 'output' folder: dashboard.html, README.md, insights.md, resume_bullets.md, images/, powerbi_data/")